In [ ]:
import numpy as np
import pandas as pd
from utils import load_results
from compare import build_comparison_df

# All instance results

In [ ]:
df = load_results({
    "up": "../results/python/results_referenced_up.json",
    "python_all": "../results/python/results_python.json",
    "vulcan_all": "../results/vulcan/results_vulcan_all.json",
})

def stats(col):
    rel = (df['baseline_cost'] - df[col]) / df['baseline_cost'] * 100
    return rel.min(), rel.mean(), rel.max()

print("==== Over ALL traces (min / avg / max % savings vs greedy) ==== ")
for label, col in [("Uniform progress", "cost_up"),
                   ("Python (all)",     "cost_python_all"),
                   ("Vulcan (all)",     "cost_vulcan_all")]:
    lo, avg, hi = stats(col)
    print(f"\t{label}: {lo:.2f}% / {avg:.2f}% / {hi:.2f}%")

In [ ]:
def dollar_stats(col):
    sav = df['baseline_cost'] - df[col]
    return sav.min(), sav.mean(), sav.max()

print("==== Over ALL traces (min / avg / max $ savings vs greedy) ==== ")
for label, col in [("Uniform progress", "cost_up"),
                   ("Python (all)",     "cost_python_all"),
                   ("Vulcan (all)",     "cost_vulcan_all")]:
    lo, avg, hi = dollar_stats(col)
    print(f"\t{label}: ${lo:.2f} / ${avg:.2f} / ${hi:.2f}")

In [ ]:
df['region'] = df['scenario'].str.split('|').str[0]

methods = {'up': 'cost_up', 'vulcan_all': 'cost_vulcan_all', 'python_all': 'cost_python_all'}

abs_sav = df.groupby('region').apply(lambda g: pd.Series({
    name: (g['baseline_cost'] - g[col]).mean() for name, col in methods.items()
}))

rel_sav = df.groupby('region').apply(lambda g: pd.Series({
    name: ((g['baseline_cost'] - g[col]) / g['baseline_cost']).mean() * 100
    for name, col in methods.items()
}))

print("==== Absolute savings ($) by region ====")
print(abs_sav.round(2))
print("\n==== Relative savings (%) by region ====")
print(rel_sav.round(2))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

regions = abs_sav.index.tolist()
x = np.arange(len(regions))
width = 0.27
colors = {'up': '#4C72B0', 'vulcan_all': '#DD8452', 'python_all': '#55A868'}

for ax, data, ylabel, title in [
    (axes[0], abs_sav, 'Savings vs greedy ($)',  'Absolute cost savings by region'),
    (axes[1], rel_sav, 'Savings vs greedy (%)',  'Relative cost savings by region'),
]:
    for i, m in enumerate(['up', 'python_all', 'vulcan_all']):
        ax.bar(x + (i - 1) * width, data[m], width, label=m, color=colors[m])
    ax.axhline(0, color='black', linewidth=0.6)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.3)

axes[1].set_xticks(x)
axes[1].set_xticklabels(regions, rotation=30, ha='right')
axes[0].legend(loc='upper left', ncol=3, frameon=True)
plt.tight_layout()
plt.show()